# Task 5 Custom Ensemble Model

## **Loading preprocessed dataset**

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load and preprocess the data
data = pd.read_csv('preprocessed_wbdc_data.csv')
X = data.drop(columns=["Diagnosis"])
y = data["Diagnosis"]

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## **Implement individual models**

In [3]:
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)  # display all columns
pd.set_option('display.expand_frame_repr', False)  # Avoiding automatic line feeds


# individual models
nb_model = GaussianNB()
svm_model = SVC(probability=True, kernel='linear', random_state=42)
dt_model = DecisionTreeClassifier(random_state=42)
rf_model = RandomForestClassifier(n_estimators=30, random_state=42)

## **Train and Evaluate Different Ensemble Voting Models**

We implemented a voting ensemble model to combine multiple classifiers and enhance predictive performance. Voting ensembles aggregate the predictions of several base models, balancing individual model weaknesses and improving overall accuracy and robustness. Here, we use soft voting, which averages the predicted probabilities from each base model and selects the class with the highest average probability. This approach leverages the strengths of diverse classifiers to produce a more reliable model.

In [4]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import time

# a function to train and evaluate Voting models
def evaluate_voting_model(estimators, model_name):
    voting_model = VotingClassifier(estimators=estimators, voting='soft')
    start_time = time.time()
    voting_model.fit(X_train, y_train)
    training_time = time.time() - start_time
    y_pred = voting_model.predict(X_test)
    return {
        'Model': model_name,
        'Training Time': training_time,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'Confusion Matrix': confusion_matrix(y_test, y_pred)
    }

In [5]:
# store evaluation results
results = []

# two-model combinations
results.append(evaluate_voting_model([('Naive Bayes', nb_model), ('SVM', svm_model)], 'Naive Bayes + SVM'))
results.append(evaluate_voting_model([('Naive Bayes', nb_model), ('Decision Tree', dt_model)], 'Naive Bayes + Decision Tree'))
results.append(evaluate_voting_model([('Naive Bayes', nb_model), ('Random Forest', rf_model)], 'Naive Bayes + Random Forest'))
results.append(evaluate_voting_model([('SVM', svm_model), ('Decision Tree', dt_model)], 'SVM + Decision Tree'))
results.append(evaluate_voting_model([('SVM', svm_model), ('Random Forest', rf_model)], 'SVM + Random Forest'))
results.append(evaluate_voting_model([('Decision Tree', dt_model), ('Random Forest', rf_model)], 'Decision Tree + Random Forest'))

# three-model combinations
results.append(evaluate_voting_model([('Naive Bayes', nb_model), ('SVM', svm_model), ('Decision Tree', dt_model)], 'Naive Bayes + SVM + Decision Tree'))
results.append(evaluate_voting_model([('Naive Bayes', nb_model), ('SVM', svm_model), ('Random Forest', rf_model)], 'Naive Bayes + SVM + Random Forest'))
results.append(evaluate_voting_model([('Naive Bayes', nb_model), ('Decision Tree', dt_model), ('Random Forest', rf_model)], 'Naive Bayes + Decision Tree + Random Forest'))
results.append(evaluate_voting_model([('SVM', svm_model), ('Decision Tree', dt_model), ('Random Forest', rf_model)], 'SVM + Decision Tree + Random Forest'))

# all four-model combination
results.append(evaluate_voting_model([('Naive Bayes', nb_model), ('SVM', svm_model), ('Decision Tree', dt_model), ('Random Forest', rf_model)], 'All Models'))

# display results
import pandas as pd
results_df = pd.DataFrame(results)
print("Voting Ensemble Comparison:")
print(results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'Training Time']])

Voting Ensemble Comparison:
                                          Model  Accuracy  Precision    Recall  F1 Score  Training Time
0                             Naive Bayes + SVM  0.941520   0.934426  0.904762  0.919355       0.016568
1                   Naive Bayes + Decision Tree  0.947368   0.909091  0.952381  0.930233       0.020663
2                   Naive Bayes + Random Forest  0.935673   0.919355  0.904762  0.912000       0.055166
3                           SVM + Decision Tree  0.941520   0.895522  0.952381  0.923077       0.014959
4                           SVM + Random Forest  0.970760   0.967742  0.952381  0.960000       0.042707
5                 Decision Tree + Random Forest  0.953216   0.923077  0.952381  0.937500       0.055226
6             Naive Bayes + SVM + Decision Tree  0.970760   0.967742  0.952381  0.960000       0.017269
7             Naive Bayes + SVM + Random Forest  0.953216   0.950820  0.920635  0.935484       0.046898
8   Naive Bayes + Decision Tree + Ra

The ensemble model enhances accuracy, precision, recall, F1 score, and confusion matrix by combining the strengths of individual models. SVM and Random Forest are the best individual performers, with SVM achieving the highest accuracy (98%) and Random Forest close behind (97.07%). While SVM alone has high precision and recall, the ensemble model further improves generalization by capturing complex patterns.

However, ensembles don't always outperform individual models; including Naive Bayes can slightly reduce performance due to its simplifying assumptions, and a standalone Decision Tree may overfit. The best ensemble configuration—SVM + Decision Tree + Random Forest—achieves 98.25% accuracy, 98.38% precision, 96.83% recall, and a 97.6% F1 score. This setup leverages SVM’s precision, Random Forest’s stability, and Decision Tree’s interpretive power to enhance overall performance.

Regarding training time, simpler combinations (e.g., Naive Bayes + SVM) are very fast (around 0.01 seconds), making them suitable for resource-limited cases. The optimal ensemble, SVM + Decision Tree + Random Forest, takes about 0.06 seconds, providing a balanced trade-off between accuracy and efficiency.

**Confusion Matrix of the best model "SVM + Decision Tree + Random Forest"**

In [6]:
import pandas as pd

best_model_name = "SVM + Decision Tree + Random Forest"
best_model_confusion_matrix = results_df[results_df['Model'] == best_model_name]['Confusion Matrix'].values[0]

# the best model's confusion matrix
print(f"Confusion Matrix for {best_model_name}:")
print(best_model_confusion_matrix)

Confusion Matrix for SVM + Decision Tree + Random Forest:
[[107   1]
 [  2  61]]


## **Train and Evaluate Different Ensemble Stacking Models**

We also implemented a stacking ensemble to enhance predictive performance by combining multiple classifiers. Like the voting ensemble, stacking leverages the strengths of diverse models to improve accuracy and robustness. However, instead of simple averaging, stacking uses a meta-learner to learn optimal combinations from the outputs of each base model, capturing more complex patterns in the data. The evaluate_stacking_model function trains and evaluates each stacking configuration on metrics such as accuracy, precision, recall, and F1-score. We tested different combinations of two, three, and four models to identify the optimal setup for improved predictive accuracy.


In [7]:
from sklearn.ensemble import StackingClassifier

# a function to to train and evaluate Stacking models
def evaluate_stacking_model(estimators, final_estimator, model_name):
    stacking_model = StackingClassifier(estimators=estimators, final_estimator=final_estimator, cv=5)
    start_time = time.time()
    stacking_model.fit(X_train, y_train)
    training_time = time.time() - start_time
    y_pred = stacking_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    return {
        'Model': model_name,
        'Training Time': training_time,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1, 
        'Confusion Matrix': confusion_matrix(y_test, y_pred)
    }

results = []

# two-model combinations
results.append(evaluate_stacking_model([('Naive Bayes', nb_model), ('SVM', svm_model)], rf_model, 'Naive Bayes + SVM'))
results.append(evaluate_stacking_model([('Naive Bayes', nb_model), ('Decision Tree', dt_model)], rf_model, 'Naive Bayes + Decision Tree'))
results.append(evaluate_stacking_model([('Naive Bayes', nb_model), ('Random Forest', rf_model)], rf_model, 'Naive Bayes + Random Forest'))
results.append(evaluate_stacking_model([('SVM', svm_model), ('Decision Tree', dt_model)], rf_model, 'SVM + Decision Tree'))
results.append(evaluate_stacking_model([('SVM', svm_model), ('Random Forest', rf_model)], rf_model, 'SVM + Random Forest'))
results.append(evaluate_stacking_model([('Decision Tree', dt_model), ('Random Forest', rf_model)], rf_model, 'Decision Tree + Random Forest'))

# three-model combinations
results.append(evaluate_stacking_model([('Naive Bayes', nb_model), ('SVM', svm_model), ('Decision Tree', dt_model)], rf_model, 'Naive Bayes + SVM + Decision Tree'))
results.append(evaluate_stacking_model([('Naive Bayes', nb_model), ('SVM', svm_model), ('Random Forest', rf_model)], rf_model, 'Naive Bayes + SVM + Random Forest'))
results.append(evaluate_stacking_model([('Naive Bayes', nb_model), ('Decision Tree', dt_model), ('Random Forest', rf_model)], rf_model, 'Naive Bayes + Decision Tree + Random Forest'))
results.append(evaluate_stacking_model([('SVM', svm_model), ('Decision Tree', dt_model), ('Random Forest', rf_model)], rf_model, 'SVM + Decision Tree + Random Forest'))

# four-model combination
results.append(evaluate_stacking_model([('Naive Bayes', nb_model), ('SVM', svm_model), ('Decision Tree', dt_model), ('Random Forest', rf_model)], rf_model, 'All Models'))

# Display results
results_df = pd.DataFrame(results)
print("Stacking Ensemble Comparison:")
print(results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'Training Time']])


Stacking Ensemble Comparison:
                                          Model  Accuracy  Precision    Recall  F1 Score  Training Time
0                             Naive Bayes + SVM  0.982456   0.982469  0.982456  0.982426       0.103786
1                   Naive Bayes + Decision Tree  0.953216   0.953173  0.953216  0.953054       0.072168
2                   Naive Bayes + Random Forest  0.964912   0.965560  0.964912  0.964656       0.297264
3                           SVM + Decision Tree  0.959064   0.959266  0.959064  0.959130       0.124970
4                           SVM + Random Forest  0.970760   0.971656  0.970760  0.970894       0.343669
5                 Decision Tree + Random Forest  0.959064   0.958996  0.959064  0.958995       0.320748
6             Naive Bayes + SVM + Decision Tree  0.964912   0.964912  0.964912  0.964912       0.124971
7             Naive Bayes + SVM + Random Forest  0.964912   0.965411  0.964912  0.965022       0.343639
8   Naive Bayes + Decision Tree + 

The stacking ensemble achieves better performance than the voting ensemble and rivals the best individual models. Unlike the voting ensemble, where Naive Bayes and SVM didn’t perform as well together, this combination excels in stacking. The stacking ensemble’s meta-learner dynamically weights each model based on data patterns, leading to more robust and accurate predictions by adapting to complex relationships. However, stacking has a higher training time—about 0.4 seconds for the optimal SVM + Decision Tree + Random Forest combination, compared to about 0.07 seconds in voting—due to the meta-learning step. Despite the extra time, stacking's accuracy gains make it valuable for resource-permitting, precision-focused applications.

**Confusion Matrix of the best model "SVM + Decision Tree + Random Forest"**

In [10]:
best_model_name = "Naive Bayes + SVM"
best_model_confusion_matrix = results_df[results_df['Model'] == best_model_name]['Confusion Matrix'].values[0]

# the best model's confusion matrix
print(f"Confusion Matrix for {best_model_name}:")
print(best_model_confusion_matrix)

Confusion Matrix for Naive Bayes + SVM:
[[107   1]
 [  2  61]]
